In [3]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# ---------- USER INPUT ----------
r_bathtub  = r"D:\Phd Research\Final_Raster\Bathtub_depth_100yr_surge_SLR.tif"
r_process  = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"
TOL = 0.01     # meters, treat depths within ±TOL as equal
SAVE_CLASS_RASTER = False
r_class_out = r"D:\Phd Research\Final_Raster\bathtub_vs_process_class.tif"
# --------------------------------

# Read process-based (reference grid)
with rasterio.open(r_process) as ref:
    proc = ref.read(1).astype(float)
    proc_nodata = ref.nodata if ref.nodata is not None else -9999.0
    ref_meta = ref.meta.copy()
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_height, ref_width = ref.height, ref.width

# Read bathtub and reproject to reference grid
with rasterio.open(r_bathtub) as src:
    bath = src.read(1).astype(float)
    bath_nodata = src.nodata if src.nodata is not None else -9999.0

    bath_on_ref = np.full((ref_height, ref_width), np.nan, dtype=float)
    reproject(
        source=bath,
        destination=bath_on_ref,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        resampling=Resampling.bilinear,  # depths: continuous
        src_nodata=bath_nodata,
        dst_nodata=np.nan
    )

# Valid overlap mask (positive depths only)
valid = (
    np.isfinite(proc) & (proc != proc_nodata) & (proc > 0) &
    np.isfinite(bath_on_ref) & (bath_on_ref > 0)
)

proc_v  = proc[valid]
bath_v  = bath_on_ref[valid]

# Classification with tolerance
diff = bath_v - proc_v
cls_bath_higher = diff >  TOL        # bathtub deeper
cls_proc_higher = diff < -TOL        # process deeper
cls_equal       = np.abs(diff) <= TOL # effectively equal

# Area weighting (if projected CRS in meters, else fall back to counts)
pixel_area_m2 = None
if ref_crs and not ref_crs.is_geographic:
    # affine: a (px width), e (px height is negative); use abs
    from affine import Affine
    a = abs(ref_transform.a)
    e = abs(ref_transform.e)
    pixel_area_m2 = a * e

def pct_area(mask):
    if pixel_area_m2 is not None:
        area_km2 = mask.sum() * pixel_area_m2 / 1e6
        total_km2 = valid.sum() * pixel_area_m2 / 1e6
        return 100.0 * area_km2 / total_km2, area_km2
    else:
        return 100.0 * mask.sum() / valid.sum(), None

p_bath, area_bath = pct_area(cls_bath_higher)
p_proc, area_proc = pct_area(cls_proc_higher)
p_eq,   area_eq   = pct_area(cls_equal)

print("=== Bathtub vs Process-based (reprojected to common grid) ===")
print(f"Valid overlapping cells: {valid.sum():,}")
if pixel_area_m2 is not None:
    print(f"Projected CRS detected; area-weighted percentages used.")
    print(f"Bathtub deeper than Process-based : {p_bath:5.2f}%  ({area_bath:.2f} km²)")
    print(f"Process-based deeper than Bathtub : {p_proc:5.2f}%  ({area_proc:.2f} km²)")
    print(f"Equal (|Δ| ≤ {TOL} m)             : {p_eq:5.2f}%  ({area_eq:.2f} km²)")
else:
    print(f"Geographic CRS detected; using cell-count percentages.")
    print(f"Bathtub deeper than Process-based : {p_bath:5.2f}%")
    print(f"Process-based deeper than Bathtub : {p_proc:5.2f}%")
    print(f"Equal (|Δ| ≤ {TOL} m)             : {p_eq:5.2f}%")

# Optional: save classification raster
if SAVE_CLASS_RASTER:
    # 1 = Bathtub deeper, -1 = Process deeper, 0 = Equal, -9999 = No data/invalid
    cls = np.full(proc.shape, -9999, dtype=np.int16)
    cls[valid] = 0
    cls_indices = np.where(valid)
    cls[cls_indices[0][cls_bath_higher], cls_indices[1][cls_bath_higher]] = 1
    cls[cls_indices[0][cls_proc_higher], cls_indices[1][cls_proc_higher]] = -1

    meta = ref_meta
    meta.update(dtype="int16", nodata=-9999, compress="lzw")
    with rasterio.open(r_class_out, "w", **meta) as dst:
        dst.write(cls, 1)
    print(f"[Saved] Classification raster → {r_class_out}")


=== Bathtub vs Process-based (reprojected to common grid) ===
Valid overlapping cells: 342,724
Projected CRS detected; area-weighted percentages used.
Bathtub deeper than Process-based : 60.43%  (8284.00 km²)
Process-based deeper than Bathtub : 38.63%  (5295.16 km²)
Equal (|Δ| ≤ 0.01 m)             :  0.95%  (129.80 km²)
